# Study Distraction Detector
## Computer Vision & AI - Semester 7

Sistem pengawasan produktivitas berbasis **Computer Vision** yang mendeteksi distraksi belajar.

**Fitur Utama:**
- Deteksi smartphone menggunakan **YOLOv8 Nano**
- Deteksi tangan menggunakan **MediaPipe Hands**
- Overlap analysis antara tangan dan smartphone
- Punishment system: **Meme visual + Suara** saat distraksi > 3 detik
- Logging screenshot distraksi

**Device:** AMD Ryzen 5 5625U | 16 GB RAM | Radeon Graphics

---
## 1. Install Dependencies

In [1]:
# Jalankan cell ini sekali untuk install semua dependency
!pip install opencv-python ultralytics mediapipe pygame numpy

  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/44.0 MB ? eta -:--:--
   ----------------------

---
## 2. Import Libraries & Konfigurasi

In [2]:
import cv2
import numpy as np
import mediapipe as mp
from ultralytics import YOLO
import pygame
import time
import os
import threading
import random
from datetime import datetime
from collections import deque

print("✅ Semua library berhasil di-import!")
print(f"OpenCV version: {cv2.__version__}")

Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Natasya\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
pygame 2.6.1 (SDL 2.28.4, Python 3.11.5)
Hello from the pygame community. https://www.pygame.org/contribute.html
✅ Semua library berhasil di-import!
OpenCV version: 5.0.0


In [3]:
# =============================================================================
# KONFIGURASI UTAMA - Sesuaikan sesuai kebutuhan
# =============================================================================

# --- Path ---
PROJECT_DIR = os.path.dirname(os.path.abspath('__file__'))
ASSETS_DIR = os.path.join(PROJECT_DIR, 'assets')
IMAGES_DIR = os.path.join(ASSETS_DIR, 'images')   # Folder meme images
SOUNDS_DIR = os.path.join(ASSETS_DIR, 'sounds')    # Folder sound effects
LOGS_DIR = os.path.join(PROJECT_DIR, 'logs', 'distractions')  # Screenshot logs

# Buat folder jika belum ada
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(SOUNDS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

# --- YOLO ---
YOLO_MODEL = 'yolov8n.pt'  # Ganti ke 'yolo11n.pt' jika ingin YOLOv11
YOLO_CONFIDENCE = 0.45     # Confidence threshold untuk deteksi
YOLO_SKIP_FRAMES = 3      # Jalankan YOLO setiap N frame (hemat CPU)

# COCO Class IDs yang relevan
COCO_CELL_PHONE = 67
COCO_BOOK = 73
COCO_LAPTOP = 63
TARGET_CLASSES = [COCO_CELL_PHONE]  # Hanya deteksi cell phone
CONTEXT_CLASSES = [COCO_BOOK, COCO_LAPTOP]  # Deteksi konteks belajar

# --- MediaPipe Hands ---
MP_MAX_HANDS = 2
MP_DETECTION_CONFIDENCE = 0.6
MP_TRACKING_CONFIDENCE = 0.5

# --- Distraction Threshold ---
DISTRACTION_TIME_THRESHOLD = 3.0  # Detik sebelum punishment dipicu

# --- Punishment ---
RED_OVERLAY_ALPHA = 0.4           # Transparansi overlay merah (0.0-1.0)
MEME_DISPLAY_SIZE = (300, 300)    # Ukuran meme di layar (width, height)
PUNISHMENT_COOLDOWN = 5.0         # Detik sebelum punishment bisa dipicu lagi

# --- Webcam ---
WEBCAM_INDEX = 0          # Index webcam (0 = default)
FRAME_WIDTH = 1280        # Resolusi lebar
FRAME_HEIGHT = 720        # Resolusi tinggi

print("✅ Konfigurasi dimuat!")
print(f"📁 Project dir: {PROJECT_DIR}")
print(f"🖼️  Meme images: {IMAGES_DIR}")
print(f"🔊 Sound files: {SOUNDS_DIR}")

✅ Konfigurasi dimuat!
📁 Project dir: d:\Kuliah\SEM 7\CV\MARIBELAJAR
🖼️  Meme images: d:\Kuliah\SEM 7\CV\MARIBELAJAR\assets\images
🔊 Sound files: d:\Kuliah\SEM 7\CV\MARIBELAJAR\assets\sounds


---
## 3. Load Model YOLOv8

In [4]:
# Load YOLO model (akan download otomatis jika belum ada)
print("⏳ Loading YOLO model...")
model = YOLO(YOLO_MODEL)
print(f"✅ Model '{YOLO_MODEL}' berhasil dimuat!")

# Verifikasi class names
print(f"\n📋 Target classes:")
print(f"   - Cell Phone (ID {COCO_CELL_PHONE}): {model.names[COCO_CELL_PHONE]}")
print(f"   - Book (ID {COCO_BOOK}): {model.names[COCO_BOOK]}")
print(f"   - Laptop (ID {COCO_LAPTOP}): {model.names[COCO_LAPTOP]}")

⏳ Loading YOLO model...
✅ Model 'yolov8n.pt' berhasil dimuat!

📋 Target classes:
   - Cell Phone (ID 67): cell phone
   - Book (ID 73): book
   - Laptop (ID 63): laptop


---
## 4. Punishment Manager (Audio + Visual)

In [5]:
class PunishmentManager:
    """Mengelola sistem punishment: audio dan visual overlay."""
    
    def __init__(self, images_dir, sounds_dir):
        self.images_dir = images_dir
        self.sounds_dir = sounds_dir
        
        # Inisialisasi pygame mixer untuk audio
        pygame.mixer.init(frequency=44100, size=-16, channels=2, buffer=512)
        
        # Load semua meme images
        self.meme_images = self._load_meme_images()
        
        # Load semua sound files
        self.sound_files = self._load_sound_files()
        
        # State
        self.is_punishing = False
        self.current_meme = None
        self.last_punishment_time = 0
        self._audio_thread = None
        
        print(f"\n🎮 Punishment Manager initialized:")
        print(f"   🖼️  Meme images found: {len(self.meme_images)}")
        print(f"   🔊 Sound files found: {len(self.sound_files)}")
        
        if len(self.meme_images) == 0:
            print(f"   ⚠️  PERINGATAN: Tidak ada gambar meme di '{images_dir}'!")
            print(f"       Taruh file .jpg/.png/.jpeg di folder tersebut.")
        if len(self.sound_files) == 0:
            print(f"   ⚠️  PERINGATAN: Tidak ada file suara di '{sounds_dir}'!")
            print(f"       Taruh file .mp3/.wav/.ogg di folder tersebut.")
    
    def _load_meme_images(self):
        """Load semua gambar meme dari folder."""
        images = []
        valid_ext = ('.jpg', '.jpeg', '.png', '.bmp')
        
        if not os.path.exists(self.images_dir):
            return images
        
        for fname in sorted(os.listdir(self.images_dir)):
            if fname.lower().endswith(valid_ext):
                fpath = os.path.join(self.images_dir, fname)
                img = cv2.imread(fpath, cv2.IMREAD_UNCHANGED)
                if img is not None:
                    # Resize ke ukuran standar
                    img = cv2.resize(img, MEME_DISPLAY_SIZE)
                    images.append({'name': fname, 'image': img})
                    print(f"   ✅ Loaded meme: {fname}")
        return images
    
    def _load_sound_files(self):
        """Load daftar path file suara dari folder."""
        sounds = []
        valid_ext = ('.mp3', '.wav', '.ogg')
        
        if not os.path.exists(self.sounds_dir):
            return sounds
        
        for fname in sorted(os.listdir(self.sounds_dir)):
            if fname.lower().endswith(valid_ext):
                fpath = os.path.join(self.sounds_dir, fname)
                sounds.append({'name': fname, 'path': fpath})
                print(f"   ✅ Loaded sound: {fname}")
        return sounds
    
    def trigger_punishment(self):
        """Memicu punishment (audio + visual)."""
        current_time = time.time()
        
        # Cek cooldown
        if current_time - self.last_punishment_time < PUNISHMENT_COOLDOWN:
            return
        
        self.is_punishing = True
        self.last_punishment_time = current_time
        
        # Pilih meme random
        if self.meme_images:
            self.current_meme = random.choice(self.meme_images)
        
        # Play audio di thread terpisah (non-blocking)
        if self.sound_files:
            self._audio_thread = threading.Thread(
                target=self._play_sound, daemon=True
            )
            self._audio_thread.start()
    
    def _play_sound(self):
        """Play sound effect di background thread."""
        try:
            sound = random.choice(self.sound_files)
            pygame.mixer.music.load(sound['path'])
            pygame.mixer.music.set_volume(0.8)
            pygame.mixer.music.play()
        except Exception as e:
            print(f"⚠️ Error playing sound: {e}")
    
    def stop_punishment(self):
        """Hentikan punishment."""
        self.is_punishing = False
        self.current_meme = None
        try:
            if pygame.mixer.music.get_busy():
                pygame.mixer.music.fadeout(500)
        except:
            pass
    
    def apply_visual_punishment(self, frame):
        """Terapkan efek visual punishment ke frame."""
        if not self.is_punishing:
            return frame
        
        h, w = frame.shape[:2]
        result = frame.copy()
        
        # --- 1. Red Overlay ---
        red_overlay = np.zeros_like(result)
        red_overlay[:, :] = (0, 0, 255)  # BGR: Red
        result = cv2.addWeighted(result, 1 - RED_OVERLAY_ALPHA, 
                                 red_overlay, RED_OVERLAY_ALPHA, 0)
        
        # --- 2. Glitch / Scanline Effect ---
        for i in range(0, h, 4):
            if random.random() > 0.7:
                shift = random.randint(-10, 10)
                result[i:i+2] = np.roll(result[i:i+2], shift, axis=1)
        
        # --- 3. Warning Text ---
        warning_text = "PUNISHMENT: JAUHKAN HP!"
        text_size = cv2.getTextSize(warning_text, cv2.FONT_HERSHEY_DUPLEX, 
                                     1.5, 3)[0]
        text_x = (w - text_size[0]) // 2
        text_y = 60
        
        # Background rectangle untuk text
        cv2.rectangle(result, (text_x - 15, text_y - 40), 
                      (text_x + text_size[0] + 15, text_y + 15), 
                      (0, 0, 0), -1)
        cv2.rectangle(result, (text_x - 15, text_y - 40), 
                      (text_x + text_size[0] + 15, text_y + 15), 
                      (0, 0, 255), 3)
        
        # Flashing text effect
        if int(time.time() * 4) % 2 == 0:
            cv2.putText(result, warning_text, (text_x, text_y),
                       cv2.FONT_HERSHEY_DUPLEX, 1.5, (0, 0, 255), 3)
        else:
            cv2.putText(result, warning_text, (text_x, text_y),
                       cv2.FONT_HERSHEY_DUPLEX, 1.5, (255, 255, 255), 3)
        
        # --- 4. Meme Image Overlay ---
        if self.current_meme is not None:
            meme_img = self.current_meme['image']
            mh, mw = meme_img.shape[:2]
            
            # Posisikan di tengah bawah frame
            x_offset = (w - mw) // 2
            y_offset = h - mh - 30
            
            # Pastikan posisi valid
            if y_offset > 0 and x_offset > 0:
                # Handle transparansi jika PNG dengan alpha channel
                if meme_img.shape[2] == 4:
                    # BGRA image
                    alpha = meme_img[:, :, 3] / 255.0
                    for c in range(3):
                        result[y_offset:y_offset+mh, x_offset:x_offset+mw, c] = \
                            (alpha * meme_img[:, :, c] + 
                             (1 - alpha) * result[y_offset:y_offset+mh, x_offset:x_offset+mw, c])
                else:
                    # BGR image
                    result[y_offset:y_offset+mh, x_offset:x_offset+mw] = meme_img
                
                # Border di sekitar meme
                cv2.rectangle(result, (x_offset-2, y_offset-2), 
                             (x_offset+mw+2, y_offset+mh+2), 
                             (0, 0, 255), 3)
                
                # Label nama meme
                meme_label = self.current_meme['name']
                cv2.putText(result, meme_label, 
                           (x_offset, y_offset - 10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, 
                           (255, 255, 255), 2)
        
        return result
    
    def cleanup(self):
        """Cleanup resources."""
        try:
            pygame.mixer.music.stop()
            pygame.mixer.quit()
        except:
            pass

print("✅ PunishmentManager class defined!")

✅ PunishmentManager class defined!


---
## 5. Detector Engine

In [6]:
class DistractionDetector:
    """Engine utama untuk deteksi distraksi belajar."""
    
    def __init__(self, yolo_model, punishment_manager):
        self.model = yolo_model
        self.punishment = punishment_manager
        
        # MediaPipe Hands
        self.mp_hands = mp.solutions.hands
        self.mp_drawing = mp.solutions.drawing_utils
        self.mp_drawing_styles = mp.solutions.drawing_styles
        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=MP_MAX_HANDS,
            min_detection_confidence=MP_DETECTION_CONFIDENCE,
            min_tracking_confidence=MP_TRACKING_CONFIDENCE
        )
        
        # Distraction state
        self.distraction_start_time = None
        self.is_distracted = False
        self.total_distraction_time = 0.0
        self.distraction_count = 0
        self.session_start_time = time.time()
        
        # YOLO frame skip
        self.frame_count = 0
        self.last_yolo_results = None
        self.last_phone_boxes = []
        self.last_context_boxes = []
        
        # FPS tracking
        self.fps_times = deque(maxlen=30)
        
        # Screenshot cooldown
        self.last_screenshot_time = 0
        self.SCREENSHOT_INTERVAL = 10.0  # Max 1 screenshot setiap 10 detik
    
    def _run_yolo(self, frame):
        """Jalankan YOLO inference pada frame."""
        results = self.model(
            frame, 
            conf=YOLO_CONFIDENCE,
            classes=TARGET_CLASSES + CONTEXT_CLASSES,
            verbose=False
        )
        
        phone_boxes = []
        context_boxes = []
        
        if results and len(results) > 0:
            for result in results:
                if result.boxes is not None:
                    for box in result.boxes:
                        cls_id = int(box.cls[0])
                        conf = float(box.conf[0])
                        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                        
                        box_info = {
                            'bbox': (x1, y1, x2, y2),
                            'class_id': cls_id,
                            'class_name': self.model.names[cls_id],
                            'confidence': conf
                        }
                        
                        if cls_id == COCO_CELL_PHONE:
                            phone_boxes.append(box_info)
                        elif cls_id in CONTEXT_CLASSES:
                            context_boxes.append(box_info)
        
        return phone_boxes, context_boxes
    
    def _get_hand_landmarks(self, frame):
        """Ekstrak hand landmarks menggunakan MediaPipe."""
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = self.hands.process(rgb_frame)
        
        hand_points = []
        
        if results.multi_hand_landmarks:
            h, w = frame.shape[:2]
            for hand_landmarks in results.multi_hand_landmarks:
                # Ambil titik-titik kunci ujung jari
                key_points = [
                    self.mp_hands.HandLandmark.INDEX_FINGER_TIP,
                    self.mp_hands.HandLandmark.THUMB_TIP,
                    self.mp_hands.HandLandmark.MIDDLE_FINGER_TIP,
                    self.mp_hands.HandLandmark.RING_FINGER_TIP,
                    self.mp_hands.HandLandmark.PINKY_TIP,
                    self.mp_hands.HandLandmark.WRIST,
                ]
                
                points = []
                all_x, all_y = [], []
                for lm_id in key_points:
                    lm = hand_landmarks.landmark[lm_id]
                    px, py = int(lm.x * w), int(lm.y * h)
                    points.append((px, py, lm_id))
                    all_x.append(px)
                    all_y.append(py)
                
                # Bounding box tangan (dari semua landmarks)
                all_lm_x = [int(lm.x * w) for lm in hand_landmarks.landmark]
                all_lm_y = [int(lm.y * h) for lm in hand_landmarks.landmark]
                hand_bbox = (
                    min(all_lm_x), min(all_lm_y),
                    max(all_lm_x), max(all_lm_y)
                )
                
                hand_points.append({
                    'fingertips': points,
                    'bbox': hand_bbox,
                    'landmarks': hand_landmarks
                })
        
        return hand_points, results
    
    def _check_overlap(self, hand_points, phone_boxes):
        """Cek apakah ada overlap antara tangan dan smartphone."""
        for hand in hand_points:
            for phone in phone_boxes:
                px1, py1, px2, py2 = phone['bbox']
                
                # Metode 1: Cek fingertip di dalam bounding box HP
                for (fx, fy, lm_id) in hand['fingertips']:
                    if px1 <= fx <= px2 and py1 <= fy <= py2:
                        return True, phone, hand
                
                # Metode 2: Cek IoU antara hand bbox dan phone bbox
                hx1, hy1, hx2, hy2 = hand['bbox']
                
                # Hitung intersection
                ix1 = max(px1, hx1)
                iy1 = max(py1, hy1)
                ix2 = min(px2, hx2)
                iy2 = min(py2, hy2)
                
                if ix1 < ix2 and iy1 < iy2:
                    intersection = (ix2 - ix1) * (iy2 - iy1)
                    phone_area = (px2 - px1) * (py2 - py1)
                    
                    # Jika overlap > 20% dari area HP
                    if phone_area > 0 and intersection / phone_area > 0.2:
                        return True, phone, hand
        
        return False, None, None
    
    def _save_screenshot(self, frame):
        """Simpan screenshot distraksi."""
        current_time = time.time()
        if current_time - self.last_screenshot_time < self.SCREENSHOT_INTERVAL:
            return
        
        self.last_screenshot_time = current_time
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"distraction_{timestamp}.jpg"
        filepath = os.path.join(LOGS_DIR, filename)
        cv2.imwrite(filepath, frame)
    
    def _draw_detections(self, frame, phone_boxes, context_boxes, 
                         hand_points, mp_results, overlap, elapsed):
        """Gambar semua deteksi pada frame."""
        h, w = frame.shape[:2]
        
        # --- Draw phone bounding boxes ---
        for phone in phone_boxes:
            x1, y1, x2, y2 = phone['bbox']
            color = (0, 0, 255) if overlap else (0, 255, 255)  # Red jika overlap
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            label = f"{phone['class_name']} {phone['confidence']:.2f}"
            cv2.putText(frame, label, (x1, y1 - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            
            # Ikon "📱" emoji substitute
            if overlap:
                cv2.putText(frame, "[!] HOLDING PHONE", (x1, y2 + 25),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
        
        # --- Draw context objects (book/laptop) ---
        for ctx in context_boxes:
            x1, y1, x2, y2 = ctx['bbox']
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            label = f"{ctx['class_name']} {ctx['confidence']:.2f}"
            cv2.putText(frame, label, (x1, y1 - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        
        # --- Draw hand landmarks ---
        if mp_results and mp_results.multi_hand_landmarks:
            for hand_landmarks in mp_results.multi_hand_landmarks:
                self.mp_drawing.draw_landmarks(
                    frame, hand_landmarks, self.mp_hands.HAND_CONNECTIONS,
                    self.mp_drawing_styles.get_default_hand_landmarks_style(),
                    self.mp_drawing_styles.get_default_hand_connections_style()
                )
        
        # --- HUD (Heads-Up Display) ---
        # Background panel
        cv2.rectangle(frame, (5, 5), (320, 145), (0, 0, 0), -1)
        cv2.rectangle(frame, (5, 5), (320, 145), (255, 255, 255), 1)
        
        # FPS
        self.fps_times.append(time.time())
        if len(self.fps_times) > 1:
            fps = len(self.fps_times) / (self.fps_times[-1] - self.fps_times[0])
        else:
            fps = 0
        cv2.putText(frame, f"FPS: {fps:.1f}", (15, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        
        # Phone detected count
        cv2.putText(frame, f"Phones: {len(phone_boxes)}", (15, 55),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
        
        # Hands detected count
        cv2.putText(frame, f"Hands: {len(hand_points)}", (15, 80),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 200, 0), 2)
        
        # Status
        if overlap:
            status_color = (0, 0, 255)
            status_text = f"DISTRACTED: {elapsed:.1f}s"
        else:
            status_color = (0, 255, 0)
            status_text = "FOCUSED"
        cv2.putText(frame, f"Status: {status_text}", (15, 105),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, status_color, 2)
        
        # Total distractions
        cv2.putText(frame, f"Distractions: {self.distraction_count}", (15, 130),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 2)
        
        # --- Progress bar menuju punishment ---
        if overlap and elapsed > 0:
            bar_width = 300
            bar_x = (w - bar_width) // 2
            bar_y = h - 30
            progress = min(elapsed / DISTRACTION_TIME_THRESHOLD, 1.0)
            
            cv2.rectangle(frame, (bar_x, bar_y), 
                         (bar_x + bar_width, bar_y + 20), (50, 50, 50), -1)
            fill_color = (0, int(255 * (1-progress)), int(255 * progress))
            cv2.rectangle(frame, (bar_x, bar_y), 
                         (bar_x + int(bar_width * progress), bar_y + 20), 
                         fill_color, -1)
            cv2.rectangle(frame, (bar_x, bar_y), 
                         (bar_x + bar_width, bar_y + 20), (255, 255, 255), 1)
            
            pct_text = f"Punishment in: {max(0, DISTRACTION_TIME_THRESHOLD - elapsed):.1f}s"
            cv2.putText(frame, pct_text, (bar_x, bar_y - 5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
        return frame
    
    def process_frame(self, frame):
        """Proses satu frame: deteksi, analisis, dan visualisasi."""
        self.frame_count += 1
        
        # --- YOLO Detection (skip frames untuk hemat CPU) ---
        if self.frame_count % YOLO_SKIP_FRAMES == 0 or self.last_yolo_results is None:
            self.last_phone_boxes, self.last_context_boxes = self._run_yolo(frame)
        
        phone_boxes = self.last_phone_boxes
        context_boxes = self.last_context_boxes
        
        # --- MediaPipe Hand Detection (setiap frame untuk responsivitas) ---
        hand_points, mp_results = self._get_hand_landmarks(frame)
        
        # --- Overlap Analysis ---
        overlap, phone_hit, hand_hit = self._check_overlap(hand_points, phone_boxes)
        
        # --- Temporal Logic ---
        elapsed = 0.0
        
        if overlap:
            if self.distraction_start_time is None:
                self.distraction_start_time = time.time()
            
            elapsed = time.time() - self.distraction_start_time
            
            # Trigger punishment setelah threshold
            if elapsed > DISTRACTION_TIME_THRESHOLD:
                if not self.is_distracted:
                    self.is_distracted = True
                    self.distraction_count += 1
                    self._save_screenshot(frame)
                
                self.punishment.trigger_punishment()
                self.total_distraction_time += (1.0 / max(len(self.fps_times), 1))
        else:
            # Reset timer dan hentikan punishment
            if self.distraction_start_time is not None:
                self.distraction_start_time = None
                self.is_distracted = False
                self.punishment.stop_punishment()
        
        # --- Draw detections ---
        frame = self._draw_detections(
            frame, phone_boxes, context_boxes, 
            hand_points, mp_results, overlap, elapsed
        )
        
        # --- Apply visual punishment ---
        frame = self.punishment.apply_visual_punishment(frame)
        
        return frame
    
    def get_session_stats(self):
        """Dapatkan statistik sesi belajar."""
        session_duration = time.time() - self.session_start_time
        return {
            'session_duration': session_duration,
            'total_distractions': self.distraction_count,
            'total_distraction_time': self.total_distraction_time,
            'focus_percentage': max(0, (1 - self.total_distraction_time / max(session_duration, 1)) * 100)
        }
    
    def cleanup(self):
        """Cleanup resources."""
        self.hands.close()
        self.punishment.cleanup()

print("✅ DistractionDetector class defined!")

✅ DistractionDetector class defined!


---
## 6. Cek File Punishment (Meme & Suara)

In [7]:
# Cek apakah file meme dan suara sudah ada
print("=" * 60)
print("📋 CEK FILE PUNISHMENT")
print("=" * 60)

print(f"\n🖼️  Folder meme: {IMAGES_DIR}")
img_files = [f for f in os.listdir(IMAGES_DIR) 
             if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
if img_files:
    for f in img_files:
        print(f"   ✅ {f}")
else:
    print("   ❌ KOSONG - Taruh gambar meme di sini!")
    print("   📌 Contoh nama file:")
    print("      - meme_1.jpg")
    print("      - meme_2.png")
    print("      - meme_3.jpg")

print(f"\n🔊 Folder suara: {SOUNDS_DIR}")
snd_files = [f for f in os.listdir(SOUNDS_DIR) 
             if f.lower().endswith(('.mp3', '.wav', '.ogg'))]
if snd_files:
    for f in snd_files:
        print(f"   ✅ {f}")
else:
    print("   ❌ KOSONG - Taruh file suara di sini!")
    print("   📌 Contoh nama file:")
    print("      - punishment_1.mp3")
    print("      - punishment_2.wav")
    print("      - punishment_3.mp3")

print("\n" + "=" * 60)

📋 CEK FILE PUNISHMENT

🖼️  Folder meme: d:\Kuliah\SEM 7\CV\MARIBELAJAR\assets\images
   ❌ KOSONG - Taruh gambar meme di sini!
   📌 Contoh nama file:
      - meme_1.jpg
      - meme_2.png
      - meme_3.jpg

🔊 Folder suara: d:\Kuliah\SEM 7\CV\MARIBELAJAR\assets\sounds
   ❌ KOSONG - Taruh file suara di sini!
   📌 Contoh nama file:
      - punishment_1.mp3
      - punishment_2.wav
      - punishment_3.mp3



---
## 7. 🚀 Jalankan Sistem Utama

In [8]:
def main():
    """Fungsi utama: jalankan Study Distraction Detector."""
    
    print("=" * 60)
    print("📵 STUDY DISTRACTION DETECTOR")
    print("=" * 60)
    print("Tekan 'Q' untuk keluar")
    print("Tekan 'S' untuk screenshot manual")
    print("Tekan 'R' untuk reset statistik")
    print("=" * 60)
    
    # --- Inisialisasi ---
    punishment_mgr = PunishmentManager(IMAGES_DIR, SOUNDS_DIR)
    detector = DistractionDetector(model, punishment_mgr)
    
    # --- Buka Webcam ---
    cap = cv2.VideoCapture(WEBCAM_INDEX)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)  # Minimize buffer delay
    
    if not cap.isOpened():
        print("❌ ERROR: Tidak bisa membuka webcam!")
        print("   Coba ganti WEBCAM_INDEX (0, 1, 2, ...)")
        return
    
    actual_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    actual_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"\n📷 Webcam opened: {actual_w}x{actual_h}")
    print("🟢 Sistem AKTIF - Mulai belajar!\n")
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("⚠️ Frame tidak terbaca, skip...")
                continue
            
            # Flip horizontal (mirror mode)
            frame = cv2.flip(frame, 1)
            
            # Proses frame
            processed_frame = detector.process_frame(frame)
            
            # Tampilkan
            cv2.imshow('Study Distraction Detector', processed_frame)
            
            # --- Keyboard Input ---
            key = cv2.waitKey(1) & 0xFF
            
            if key == ord('q') or key == ord('Q'):
                print("\n👋 Keluar dari sistem...")
                break
            elif key == ord('s') or key == ord('S'):
                timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                filename = f"manual_{timestamp}.jpg"
                filepath = os.path.join(LOGS_DIR, filename)
                cv2.imwrite(filepath, frame)
                print(f"📸 Screenshot saved: {filename}")
            elif key == ord('r') or key == ord('R'):
                detector.distraction_count = 0
                detector.total_distraction_time = 0.0
                detector.session_start_time = time.time()
                print("🔄 Statistik direset!")
    
    except KeyboardInterrupt:
        print("\n⏹️ Dihentikan oleh user.")
    
    finally:
        # --- Session Summary ---
        stats = detector.get_session_stats()
        print("\n" + "=" * 60)
        print("📊 SESSION SUMMARY")
        print("=" * 60)
        duration_min = stats['session_duration'] / 60
        print(f"⏱️  Durasi sesi     : {duration_min:.1f} menit")
        print(f"📵 Total distraksi  : {stats['total_distractions']} kali")
        print(f"⏳ Waktu distraksi  : {stats['total_distraction_time']:.1f} detik")
        print(f"🎯 Fokus            : {stats['focus_percentage']:.1f}%")
        print("=" * 60)
        
        # Cleanup
        detector.cleanup()
        cap.release()
        cv2.destroyAllWindows()
        print("\n✅ Cleanup selesai. Selamat belajar! 📚")

print("✅ Fungsi main() siap! Jalankan cell berikutnya untuk memulai.")

✅ Fungsi main() siap! Jalankan cell berikutnya untuk memulai.


In [9]:
# ============================================
# 🚀 JALANKAN SISTEM DI SINI
# ============================================
# Pastikan webcam tidak digunakan aplikasi lain!
# Tekan 'Q' di window OpenCV untuk berhenti.

main()

📵 STUDY DISTRACTION DETECTOR
Tekan 'Q' untuk keluar
Tekan 'S' untuk screenshot manual
Tekan 'R' untuk reset statistik

🎮 Punishment Manager initialized:
   🖼️  Meme images found: 0
   🔊 Sound files found: 0
   ⚠️  PERINGATAN: Tidak ada gambar meme di 'd:\Kuliah\SEM 7\CV\MARIBELAJAR\assets\images'!
       Taruh file .jpg/.png/.jpeg di folder tersebut.
   ⚠️  PERINGATAN: Tidak ada file suara di 'd:\Kuliah\SEM 7\CV\MARIBELAJAR\assets\sounds'!
       Taruh file .mp3/.wav/.ogg di folder tersebut.


AttributeError: module 'mediapipe' has no attribute 'solutions'

---
## 8. (Bonus) Quick Test - Deteksi pada Gambar Statis

In [ ]:
# Test YOLO detection pada satu gambar (opsional)
# Berguna untuk verifikasi model berjalan sebelum pakai webcam

def test_on_image(image_path=None):
    """Test deteksi pada gambar statis."""
    if image_path and os.path.exists(image_path):
        img = cv2.imread(image_path)
    else:
        # Buat dummy image untuk test
        print("Menggunakan dummy frame dari webcam...")
        cap = cv2.VideoCapture(WEBCAM_INDEX)
        ret, img = cap.read()
        cap.release()
        if not ret:
            print("❌ Tidak bisa baca webcam untuk test.")
            return
    
    # Run YOLO
    results = model(img, conf=YOLO_CONFIDENCE, verbose=False)
    
    # Annotate
    annotated = results[0].plot()
    
    # Simpan hasil
    output_path = os.path.join(LOGS_DIR, 'test_detection.jpg')
    cv2.imwrite(output_path, annotated)
    print(f"\n✅ Hasil test disimpan: {output_path}")
    
    # Tampilkan deteksi
    if results[0].boxes:
        for box in results[0].boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            name = model.names[cls_id]
            print(f"   Detected: {name} (conf: {conf:.2f})")
    else:
        print("   Tidak ada objek terdeteksi.")

# Uncomment untuk test:
# test_on_image()